In [ ]:
!pip -q install pandas numpy requests scikit-learn tensorflow==2.15.0 tqdm pyarrow google-cloud-storage

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0)
ERROR: No matching distribution found for tensorflow==2.15.0


In [ ]:
import time, json, requests
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error

import tensorflow as tf
print("TF:", tf.__version__)

BASE_URL = "https://api.openf1.org/v1"

# More data: pull all races since 2023 (free tier supports historical 2023+) :contentReference[oaicite:6]{index=6}
YEARS = [2023, 2024, 2025, 2026]   # if 2026 has data, it will be included; otherwise it will just return empty
SESSION_NAME = "Race"

# Pit window: classes 0..K-1 for pit in 0..K-1 laps, plus class K for "no pit within K laps"
K_WINDOW = 10

# OpenF1 rate limits => be polite + cache :contentReference[oaicite:7]{index=7}
SLEEP = 2.1
CACHE_DIR = Path("/content/openf1_cache"); CACHE_DIR.mkdir(exist_ok=True)

MODEL_DIR = Path("/content/models"); MODEL_DIR.mkdir(exist_ok=True)
PIT_MODEL_PATH = str(MODEL_DIR / "pit_window_model.h5")
DEG_MODEL_PATH = str(MODEL_DIR / "deg_model.h5")

TF: 2.19.0


In [ ]:
def cache_path(endpoint: str, params: dict) -> Path:
    safe = endpoint + "__" + "__".join([f"{k}={params[k]}" for k in sorted(params.keys())])
    safe = safe.replace("/", "_").replace(" ", "_")
    return CACHE_DIR / f"{safe}.json"

def get_openf1(endpoint: str, params: dict, retries: int = 3):
    p = cache_path(endpoint, params)
    if p.exists():
        return json.loads(p.read_text())

    url = f"{BASE_URL}/{endpoint}"
    last_err = None

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=60)

            # OpenF1 uses 404 to mean "no rows"
            if r.status_code == 404:
                data = []
                p.write_text(json.dumps(data))
                time.sleep(SLEEP)
                return data

            if r.status_code != 200:
                raise RuntimeError(f"{endpoint} -> {r.status_code}: {r.text[:200]}")

            data = r.json()
            p.write_text(json.dumps(data))
            time.sleep(SLEEP)
            return data

        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))

    raise RuntimeError(f"Failed: {endpoint} {params}. Last error: {last_err}")

In [ ]:
def list_race_sessions(year: int):
    sessions = get_openf1("sessions", {"year": year, "session_name": SESSION_NAME})
    sessions = sorted(sessions, key=lambda x: x.get("date_start", ""))
    return sessions

all_sessions = []
for y in YEARS:
    all_sessions += list_race_sessions(y)

sessions_df = pd.DataFrame(all_sessions)
print("Race sessions found:", len(sessions_df))
display(sessions_df[["year","country_name","circuit_short_name","session_key","date_start"]].head(30))

session_keys = sessions_df["session_key"].astype(int).tolist()

# helpful map for circuit/country
session_meta = sessions_df.set_index("session_key")[["year","country_name","circuit_short_name"]].to_dict("index")

Race sessions found: 95


,year,country_name,circuit_short_name,session_key,date_start
0,2023,Bahrain,Sakhir,7953,2023-03-05T15:00:00+00:00
1,2023,Saudi Arabia,Jeddah,7779,2023-03-19T17:00:00+00:00
2,2023,Australia,Melbourne,7787,2023-04-02T05:00:00+00:00
3,2023,Azerbaijan,Baku,9070,2023-04-30T11:00:00+00:00
4,2023,United States,Miami,9078,2023-05-07T19:30:00+00:00
5,2023,Italy,Imola,9086,2023-05-21T13:00:00+00:00
6,2023,Monaco,Monte Carlo,9094,2023-05-28T13:00:00+00:00
7,2023,Spain,Catalunya,9102,2023-06-04T13:00:00+00:00
8,2023,Canada,Montreal,9110,2023-06-18T18:00:00+00:00
9,2023,Austria,Spielberg,9118,2023-07-02T13:00:00+00:00


In [ ]:
def snapshot_session(session_key: int):
    # core
    laps      = get_openf1("laps",    {"session_key": session_key})
    stints    = get_openf1("stints",  {"session_key": session_key})
    drivers   = get_openf1("drivers", {"session_key": session_key})

    # context
    intervals = get_openf1("intervals", {"session_key": session_key})      # gap_to_leader, interval :contentReference[oaicite:8]{index=8}
    weather   = get_openf1("weather",   {"session_key": session_key})      # track_temperature, rainfall, etc :contentReference[oaicite:9]{index=9}
    race_ctrl = get_openf1("race_control", {"session_key": session_key})   # category/flag/lap_number :contentReference[oaicite:10]{index=10}

    return laps, stints, drivers, intervals, weather, race_ctrl

snapshots = {}
skipped = []

for sk in tqdm(session_keys, desc="Downloading sessions"):
    sk = int(sk)
    laps, stints, drivers, intervals, weather, race_ctrl = snapshot_session(sk)

    if len(laps) == 0 or len(stints) == 0 or len(drivers) == 0:
        skipped.append(sk)
        continue

    snapshots[sk] = {
        "laps": laps,
        "stints": stints,
        "drivers": drivers,
        "intervals": intervals,
        "weather": weather,
        "race_control": race_ctrl,
        "meta": session_meta.get(sk, {})
    }

print("Kept sessions:", len(snapshots))
print("Skipped:", len(skipped), "example:", skipped[:5])

Kept sessions: 70
Skipped: 25 example: [9086, 11234, 11245, 11253, 11261]


In [ ]:
import numpy as np
import pandas as pd

def _to_dt(s):
    return pd.to_datetime(s, utc=True, errors="coerce")

def _gap_to_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        t = x.strip()
        if t == "" or t.lower() == "null":
            return np.nan
        if "LAP" in t.upper():
            return 99.0
        if t.startswith("+"):
            t = t[1:]
        try:
            return float(t)
        except:
            return np.nan
    return np.nan

def build_session_dataframe(pack, k_window=10):
    laps = pd.DataFrame(pack.get("laps", []))
    stints_raw = pd.DataFrame(pack.get("stints", []))
    drivers = pd.DataFrame(pack.get("drivers", []))
    intervals = pd.DataFrame(pack.get("intervals", []))
    weather = pd.DataFrame(pack.get("weather", []))
    rc = pd.DataFrame(pack.get("race_control", []))

    if laps.empty or stints_raw.empty or drivers.empty:
        return pd.DataFrame()

    # --- Prepare stints (rename to avoid collisions)
    stints = stints_raw.copy()
    stints = stints.rename(columns={
        "lap_start": "stint_lap_start",
        "lap_end": "stint_lap_end",
        "compound": "stint_compound",
        "tyre_age_at_start": "stint_tyre_age_at_start"
    })

    # --- Laps essentials
    must = ["driver_number", "lap_number", "lap_duration", "date_start"]
    for c in must:
        if c not in laps.columns:
            return pd.DataFrame()

    laps = laps.copy()
    laps["date_start"] = _to_dt(laps["date_start"])
    laps["lap_duration"] = pd.to_numeric(laps["lap_duration"], errors="coerce")
    laps = laps.dropna(subset=["date_start", "lap_duration"])

    # sectors
    for c in ["duration_sector_1","duration_sector_2","duration_sector_3"]:
        if c not in laps.columns:
            laps[c] = np.nan
        laps[c] = pd.to_numeric(laps[c], errors="coerce")

    # --- Join laps to stints (for stint_age + tyre compound/age)
    join_keys = [k for k in ["session_key","meeting_key","driver_number"] if k in laps.columns and k in stints.columns]
    if not join_keys:
        join_keys = ["driver_number"]

    merged = laps.merge(stints, on=join_keys, how="left")
    merged = merged[
        (merged["lap_number"] >= merged["stint_lap_start"]) &
        (merged["lap_number"] <= merged["stint_lap_end"])
    ].copy()

    merged["stint_age"] = merged["lap_number"] - merged["stint_lap_start"]

    # ✅ Compound + tyre_age from STINTS (authoritative) :contentReference[oaicite:1]{index=1}
    merged["compound"] = merged.get("stint_compound", None)
    if merged["compound"] is None:
        merged["compound"] = "UNKNOWN"
    merged["compound"] = merged["compound"].fillna("UNKNOWN").astype(str)

    merged["tyre_age_at_start"] = pd.to_numeric(merged.get("stint_tyre_age_at_start", 0.0), errors="coerce").fillna(0.0)

    # --- Sort per driver/lap
    merged = merged.sort_values(["driver_number", "lap_number"]).reset_index(drop=True)

    # ✅ Use previous lap values (more realistic, avoids “pit lap duration” leakage)
    merged["lap_duration_prev"] = merged.groupby("driver_number")["lap_duration"].shift(1)
    merged["sector1_prev"] = merged.groupby("driver_number")["duration_sector_1"].shift(1)
    merged["sector2_prev"] = merged.groupby("driver_number")["duration_sector_2"].shift(1)
    merged["sector3_prev"] = merged.groupby("driver_number")["duration_sector_3"].shift(1)

    merged["lap_duration_prev"] = merged["lap_duration_prev"].fillna(merged["lap_duration"])
    merged["sector1_prev"] = merged["sector1_prev"].fillna(0.0)
    merged["sector2_prev"] = merged["sector2_prev"].fillna(0.0)
    merged["sector3_prev"] = merged["sector3_prev"].fillna(0.0)

    # Pace trend based on past laps (already past-only)
    merged["lap_mean_last3"] = merged.groupby("driver_number")["lap_duration"].transform(
        lambda s: s.shift(1).rolling(3).mean()
    ).fillna(merged["lap_duration_prev"])

    def slope_last3(s):
        def _slope(v):
            if len(v) != 3:
                return np.nan
            return np.polyfit([0,1,2], v, 1)[0]
        return s.shift(1).rolling(3).apply(_slope, raw=False)

    merged["lap_slope_last3"] = merged.groupby("driver_number")["lap_duration"].transform(slope_last3).fillna(0.0)

    # --- Drivers (team/acronym)
    dcols = ["driver_number"]
    if "team_name" in drivers.columns: dcols.append("team_name")
    if "name_acronym" in drivers.columns: dcols.append("name_acronym")
    drivers_small = drivers[dcols].copy()
    if "team_name" not in drivers_small.columns:
        drivers_small["team_name"] = "UNKNOWN"
    if "name_acronym" not in drivers_small.columns:
        drivers_small["name_acronym"] = "UNK"
    drivers_small["team_name"] = drivers_small["team_name"].fillna("UNKNOWN").astype(str)
    drivers_small["name_acronym"] = drivers_small["name_acronym"].fillna("UNK").astype(str)
    merged = merged.merge(drivers_small, on="driver_number", how="left")

    # --- Intervals (robust, rename RHS to avoid collisions)
    merged["gap_to_leader"] = 0.0
    merged["interval_ahead"] = 0.0
    if not intervals.empty and "date" in intervals.columns and "driver_number" in intervals.columns:
        intervals = intervals.copy()
        intervals["date"] = _to_dt(intervals["date"])
        intervals = intervals.dropna(subset=["date"])

        intervals["driver_number"] = pd.to_numeric(intervals["driver_number"], errors="coerce")
        intervals = intervals.dropna(subset=["driver_number"])
        intervals["driver_number"] = intervals["driver_number"].astype(int)

        if "gap_to_leader" not in intervals.columns:
            intervals["gap_to_leader"] = np.nan
        if "interval" not in intervals.columns:
            intervals["interval"] = np.nan

        intervals["gap_to_leader_iv"] = intervals["gap_to_leader"].map(_gap_to_float)
        intervals["interval_ahead_iv"] = intervals["interval"].map(_gap_to_float)

        rhs = intervals[["driver_number","date","gap_to_leader_iv","interval_ahead_iv"]].sort_values(["date","driver_number"]).reset_index(drop=True)

        merged = merged.dropna(subset=["date_start"]).copy()
        merged["driver_number"] = pd.to_numeric(merged["driver_number"], errors="coerce").fillna(-1).astype(int)
        merged = merged.sort_values(["date_start","driver_number"]).reset_index(drop=True)

        merged = pd.merge_asof(
            merged,
            rhs,
            left_on="date_start",
            right_on="date",
            by="driver_number",
            direction="backward",
            allow_exact_matches=True
        )
        merged["gap_to_leader"] = pd.to_numeric(merged["gap_to_leader_iv"], errors="coerce").fillna(0.0)
        merged["interval_ahead"] = pd.to_numeric(merged["interval_ahead_iv"], errors="coerce").fillna(0.0)
        merged = merged.drop(columns=["date","gap_to_leader_iv","interval_ahead_iv"], errors="ignore")

    # --- Weather (rename RHS)
    merged["track_temperature"] = 0.0
    merged["rainfall"] = 0.0
    merged["wind_speed"] = 0.0
    if not weather.empty and "date" in weather.columns:
        weather = weather.copy()
        weather["date"] = _to_dt(weather["date"])
        weather = weather.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

        for c in ["track_temperature","rainfall","wind_speed"]:
            if c not in weather.columns:
                weather[c] = 0.0
            weather[c] = pd.to_numeric(weather[c], errors="coerce").fillna(0.0)

        rhs_w = weather[["date","track_temperature","rainfall","wind_speed"]].rename(
            columns={"track_temperature":"track_temperature_w","rainfall":"rainfall_w","wind_speed":"wind_speed_w"}
        )

        merged = merged.sort_values("date_start").reset_index(drop=True)
        merged = pd.merge_asof(
            merged,
            rhs_w.sort_values("date"),
            left_on="date_start",
            right_on="date",
            direction="backward",
            allow_exact_matches=True
        )
        merged["track_temperature"] = merged["track_temperature_w"].fillna(0.0)
        merged["rainfall"] = merged["rainfall_w"].fillna(0.0)
        merged["wind_speed"] = merged["wind_speed_w"].fillna(0.0)
        merged = merged.drop(columns=["date","track_temperature_w","rainfall_w","wind_speed_w"], errors="ignore")

    # --- Race control flags
    merged["flag_yellow"] = 0
    merged["flag_safetycar"] = 0
    if not rc.empty and "lap_number" in rc.columns:
        rc = rc.copy()
        rc["lap_number"] = pd.to_numeric(rc["lap_number"], errors="coerce")
        rc = rc.dropna(subset=["lap_number"])
        rc["lap_number"] = rc["lap_number"].astype(int)
        rc["flag"] = rc.get("flag", "").fillna("").astype(str)
        rc["category"] = rc.get("category", "").fillna("").astype(str)

        yellow_laps = set(rc.loc[rc["flag"].str.contains("YELLOW", case=False, na=False), "lap_number"].tolist())
        sc_laps = set(rc.loc[rc["category"].str.contains("SafetyCar", case=False, na=False), "lap_number"].tolist())
        merged["flag_yellow"] = merged["lap_number"].astype(int).isin(yellow_laps).astype(int)
        merged["flag_safetycar"] = merged["lap_number"].astype(int).isin(sc_laps).astype(int)

    # --- Meta
    meta = pack.get("meta", {})
    merged["circuit_short_name"] = str(meta.get("circuit_short_name", "UNKNOWN"))
    merged["country_name"] = str(meta.get("country_name", "UNKNOWN"))
    merged["year"] = int(meta.get("year", 0)) if meta.get("year") is not None else 0

    # --- Labels
    delta = (merged["stint_lap_end"] - merged["lap_number"]).astype(int)
    y = np.where(delta < 0, k_window, delta)
    y = np.clip(y, 0, k_window)
    merged["y_pit_class"] = y.astype(int)

    merged = merged.sort_values(["driver_number","lap_number"]).reset_index(drop=True)
    merged["y_next_lap_duration"] = merged.groupby("driver_number")["lap_duration"].shift(-1)
    merged["y_next_lap_duration"] = pd.to_numeric(merged["y_next_lap_duration"], errors="coerce")

    return merged.reset_index(drop=True)

print("✅ Updated builder: compound from stints + prev-lap features (leakage reduced)")

✅ Updated builder: compound from stints + prev-lap features (leakage reduced)


In [ ]:
from tqdm import tqdm
import pandas as pd

all_rows = []
for sk, pack in tqdm(snapshots.items(), desc="Rebuilding merged rows"):
    df = build_session_dataframe(pack, k_window=K_WINDOW)
    if not df.empty:
        all_rows.append(df)

full = pd.concat(all_rows, ignore_index=True)

print("Rows:", len(full))
print("Unique compounds:", full["compound"].nunique())
print(full["compound"].value_counts().head(10))

Rebuilding merged rows: 100%|██████████| 70/70 [00:24<00:00,  2.87it/s]

Rows: 77307
Unique compounds: 5
compound
HARD            36504
MEDIUM          27880
SOFT             8535
INTERMEDIATE     4301
WET                87
Name: count, dtype: int64



/tmp/ipython-input-169/2653610451.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full = pd.concat(all_rows, ignore_index=True)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

K = K_WINDOW  # shorthand

# Targets
PIT_CLASS = "y_pit_class"  # 0..K (K means "no pit within K")
PIT_BIN   = "y_pit_within" # 1 if <K else 0

# Use your fixed columns from builder:
# - compound is now from stints ✅
# - lap_duration_prev / sector*_prev exist ✅

CAT_COLS = ["compound","team_name","name_acronym","circuit_short_name","country_name"]

NUM_COLS = [
    "tyre_age_at_start",
    "stint_age",
    "lap_duration_prev",
    "lap_mean_last3",
    "lap_slope_last3",
    "sector1_prev","sector2_prev","sector3_prev",
    "gap_to_leader","interval_ahead",
    "track_temperature","rainfall","wind_speed",
    "flag_yellow","flag_safetycar",
    "lap_number",
    "year"
]

# Ensure columns exist and are clean
df = full.copy()

# Create binary target
df[PIT_BIN] = (df[PIT_CLASS].astype(int) < K).astype(int)

for c in CAT_COLS:
    if c not in df.columns:
        df[c] = "UNKNOWN"
    df[c] = df[c].fillna("UNKNOWN").astype(str)

for n in NUM_COLS:
    if n not in df.columns:
        df[n] = 0.0
    df[n] = pd.to_numeric(df[n], errors="coerce").fillna(0.0)

df["session_key"] = pd.to_numeric(df["session_key"], errors="coerce").fillna(0).astype(int)

# Split by session_key to avoid leakage across same race
X = df[["session_key"] + CAT_COLS + NUM_COLS].copy()
y = df[PIT_BIN].copy()
groups = df["session_key"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("Train pit-within rate:", train_df[PIT_BIN].mean(), "Test:", test_df[PIT_BIN].mean())
print("Compounds (train):")
print(train_df["compound"].value_counts().head(10))

Train rows: 61078 Test rows: 16229
Train pit-within rate: 0.4387340777366646 Test: 0.44691601454186947
Compounds (train):
compound
HARD            28195
MEDIUM          22727
SOFT             6558
INTERMEDIATE     3561
WET                37
Name: count, dtype: int64


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def make_ds_binary(df_in, batch=2048, shuffle=True):
    df2 = df_in.copy()
    y = df2.pop(PIT_BIN).values.astype(np.float32)

    x = {}
    for c in CAT_COLS:
        x[c] = df2[c].astype(str).values
    for n in NUM_COLS:
        x[n] = df2[n].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df2), 30000), seed=42)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

bin_train = train_df[CAT_COLS + NUM_COLS + [PIT_BIN]].copy()
bin_test  = test_df[CAT_COLS + NUM_COLS + [PIT_BIN]].copy()

train_ds = make_ds_binary(bin_train, shuffle=True)
test_ds  = make_ds_binary(bin_test, shuffle=False)

# class weights for imbalance
pos = float(bin_train[PIT_BIN].mean())
neg = 1.0 - pos
# weight positives higher
class_weight = {0: 1.0, 1: (neg / (pos + 1e-9))}
print("Binary class_weight:", class_weight)

# Inputs
inputs = {}
for c in CAT_COLS:
    inputs[c] = keras.Input(shape=(1,), name=c, dtype=tf.string)
for n in NUM_COLS:
    inputs[n] = keras.Input(shape=(1,), name=n, dtype=tf.float32)

# Encoders
cat_vecs = []
for c in CAT_COLS:
    lk = keras.layers.StringLookup(output_mode="one_hot", name=f"lk_{c}")
    lk.adapt(bin_train[c].astype(str).values)
    cat_vecs.append(lk(inputs[c]))

num_vecs = []
for n in NUM_COLS:
    norm = keras.layers.Normalization(name=f"norm_{n}")
    norm.adapt(bin_train[n].astype(np.float32).values.reshape(-1,1))
    num_vecs.append(norm(inputs[n]))

x = keras.layers.Concatenate()(cat_vecs + num_vecs)
x = keras.layers.Dense(192, activation="relu")(x)
x = keras.layers.Dropout(0.25)(x)
x = keras.layers.Dense(96, activation="relu")(x)
x = keras.layers.Dropout(0.25)(x)
x = keras.layers.Dense(64, activation="relu")(x)
out = keras.layers.Dense(1, activation="sigmoid")(x)

pit_within_model = keras.Model(inputs=inputs, outputs=out)
pit_within_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(name="auc"),
        keras.metrics.AUC(curve="PR", name="pr_auc")
    ]
)
pit_within_model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_pr_auc", patience=3, mode="max", restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", patience=2, factor=0.5, mode="max", min_lr=1e-5),
]

hist_bin = pit_within_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

PIT_WITHIN_PATH = "/content/models/pit_within_k.keras"
pit_within_model.save(PIT_WITHIN_PATH)
print("✅ Saved:", PIT_WITHIN_PATH)

Binary class_weight: {0: 1.0, 1: 1.2792849916730913}


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ compound            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ team_name           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ name_acronym        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ circuit_short_name  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ country_name        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tyre_age_at_start   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stint_age           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lap_duration_prev   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lap_mean_last3      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lap_slope_last3     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sector1_prev        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sector2_prev        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sector3_prev        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap_to_leader       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ interval_ahead      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ track_temperature   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rainfall            │ (None, 1)         │          0 │ -               

 Total params: 46,740 (182.64 KB)

 Trainable params: 46,689 (182.38 KB)

 Non-trainable params: 51 (272.00 B)

Epoch 1/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 119ms/step - auc: 0.6452 - loss: 0.7394 - pr_auc: 0.5896 - val_auc: 0.7811 - val_loss: 0.5706 - val_pr_auc: 0.7452 - learning_rate: 0.0010
Epoch 2/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 117ms/step - auc: 0.7797 - loss: 0.6379 - pr_auc: 0.7247 - val_auc: 0.7966 - val_loss: 0.5481 - val_pr_auc: 0.7544 - learning_rate: 0.0010
Epoch 3/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 83ms/step - auc: 0.8257 - loss: 0.5796 - pr_auc: 0.7912 - val_auc: 0.7973 - val_loss: 0.5478 - val_pr_auc: 0.7590 - learning_rate: 0.0010
Epoch 4/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - auc: 0.8499 - loss: 0.5421 - pr_auc: 0.8272 - val_auc: 0.7987 - val_loss: 0.5527 - val_pr_auc: 0.7674 - learning_rate: 0.0010
Epoch 5/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - auc: 0.8701 - loss: 0.5073 - pr_auc: 0.8517 - val_auc: 0.8063 - val_loss: 0.5470 - val_pr_auc: 0.7774 - learning_rate: 0.0010
Epoch 6/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 100ms/step - auc: 0.8831 - loss: 0.4835 - pr_auc: 0.8660 - val

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

def predict_bin(df_in):
    x = {}
    for c in CAT_COLS:
        x[c] = df_in[c].astype(str).values
    for n in NUM_COLS:
        x[n] = df_in[n].astype(np.float32).values
    return pit_within_model.predict(x, batch_size=4096, verbose=0).reshape(-1)

y_true = test_df[PIT_BIN].values.astype(int)
p = predict_bin(test_df)

print("TEST ROC-AUC:", roc_auc_score(y_true, p))
print("TEST PR-AUC :", average_precision_score(y_true, p))

TEST ROC-AUC: 0.8278673042171008
TEST PR-AUC : 0.8336487526118976


In [ ]:
import numpy as np

def pr_at_thr(y, p, thr):
    yhat = (p >= thr).astype(int)
    tp = ((yhat==1)&(y==1)).sum()
    fp = ((yhat==1)&(y==0)).sum()
    fn = ((yhat==0)&(y==1)).sum()
    precision = tp/(tp+fp+1e-9)
    recall = tp/(tp+fn+1e-9)
    return precision, recall, int(tp), int(fp), int(fn)

for thr in [0.3,0.4,0.5,0.6,0.7]:
    pr, rc, tp, fp, fn = pr_at_thr(y_true, p, thr)
    print(f"thr={thr:.1f}  precision={pr:.3f}  recall={rc:.3f}  tp={tp} fp={fp} fn={fn}")

thr=0.3  precision=0.719  recall=0.767  tp=5566 fp=2174 fn=1687
thr=0.4  precision=0.751  recall=0.737  tp=5349 fp=1777 fn=1904
thr=0.5  precision=0.779  recall=0.712  tp=5167 fp=1465 fn=2086
thr=0.6  precision=0.801  recall=0.678  tp=4918 fp=1220 fn=2335
thr=0.7  precision=0.823  recall=0.640  tp=4644 fp=998 fn=2609


In [ ]:
import numpy as np
import pandas as pd

K = K_WINDOW
df = full.copy()

# Ensure numeric
df["driver_number"] = pd.to_numeric(df["driver_number"], errors="coerce").fillna(-1).astype(int)
df["lap_number"] = pd.to_numeric(df["lap_number"], errors="coerce").fillna(-1).astype(int)
df["stint_lap_end"] = pd.to_numeric(df["stint_lap_end"], errors="coerce").fillna(-1).astype(int)

# Driver max lap within a session (safest: include session_key)
df["driver_max_lap"] = df.groupby(["session_key","driver_number"])["lap_number"].transform("max").astype(int)

# If stint ends at the driver's final lap, it's race end, NOT a pit
is_race_end = (df["stint_lap_end"] >= df["driver_max_lap"])

delta = (df["stint_lap_end"] - df["lap_number"]).astype(int)
delta = np.where(is_race_end, K, delta)      # race-end stints => no pit in window
delta = np.clip(delta, 0, K)

df["y_pit_class"] = delta.astype(int)
df["y_pit_within"] = (df["y_pit_class"] < K).astype(int)

full = df  # overwrite global

print("✅ Relabeled with race-end fix.")
print("New pit-within rate:", full["y_pit_within"].mean())
print("Class distribution (0..K):")
print(full["y_pit_class"].value_counts(normalize=True).sort_index().head(12))

✅ Relabeled with race-end fix.
New pit-within rate: 0.278305974879377
Class distribution (0..K):
y_pit_class
0     0.032869
1     0.030023
2     0.028665
3     0.028186
4     0.027850
5     0.027449
6     0.027151
7     0.026143
8     0.025366
9     0.024603
10    0.721694
Name: proportion, dtype: float64


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

K = K_WINDOW
PIT_CLASS = "y_pit_class"
PIT_BIN = "y_pit_within"

# ---- Build conditional train/test (assumes full/train_df/test_df already exist from the relabel step)
cond_train = train_df[(train_df[PIT_BIN] == 1) & (train_df[PIT_CLASS] < K)].copy()
cond_test  = test_df[(test_df[PIT_BIN] == 1) & (test_df[PIT_CLASS] < K)].copy()

cond_train[PIT_CLASS] = cond_train[PIT_CLASS].astype(int)
cond_test[PIT_CLASS]  = cond_test[PIT_CLASS].astype(int)

print("Conditional train rows:", len(cond_train), "Conditional test rows:", len(cond_test))

def make_ds_cond(df_in, batch=512, shuffle=True):
    df2 = df_in.copy()
    y = df2.pop(PIT_CLASS).values.astype(np.int32)

    x = {}
    for c in CAT_COLS:
        x[c] = df2[c].astype(str).values
    for n in NUM_COLS:
        x[n] = df2[n].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df2), 30000), seed=42)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds_cond(cond_train[CAT_COLS + NUM_COLS + [PIT_CLASS]], shuffle=True)
test_ds  = make_ds_cond(cond_test[CAT_COLS + NUM_COLS + [PIT_CLASS]], shuffle=False)

# Class weights 0..K-1
counts = cond_train[PIT_CLASS].value_counts().sort_index()
freq = counts / counts.sum()
w = (1.0 / (freq + 1e-9))
w = (w / w.mean()).clip(0.3, 6.0)
class_weight = {int(k): float(v) for k, v in w.items()}
print("Class weight sample:", list(class_weight.items())[:10])

# ---- Model
inputs = {}
for c in CAT_COLS:
    inputs[c] = keras.Input(shape=(1,), name=c, dtype=tf.string)
for n in NUM_COLS:
    inputs[n] = keras.Input(shape=(1,), name=n, dtype=tf.float32)

cat_vecs = []
for c in CAT_COLS:
    lk = keras.layers.StringLookup(output_mode="one_hot", name=f"lk_{c}")
    lk.adapt(cond_train[c].astype(str).values)
    cat_vecs.append(lk(inputs[c]))

num_vecs = []
for n in NUM_COLS:
    norm = keras.layers.Normalization(name=f"norm_{n}")
    norm.adapt(cond_train[n].astype(np.float32).values.reshape(-1,1))
    num_vecs.append(norm(inputs[n]))

x = keras.layers.Concatenate()(cat_vecs + num_vecs)
x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.25)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.25)(x)
x = keras.layers.Dense(64, activation="relu")(x)
out = keras.layers.Dense(K, activation="softmax")(x)

pit_when_model = keras.Model(inputs=inputs, outputs=out)

# ✅ No label_smoothing here (not supported for SparseCategoricalCrossentropy)
pit_when_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_acc", patience=4, mode="max", restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_acc", patience=2, factor=0.5, mode="max", min_lr=1e-5),
]

hist_cond = pit_when_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=25,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

PIT_WHEN_PATH = "/content/models/pit_when_0_to_kminus1.keras"
pit_when_model.save(PIT_WHEN_PATH)
print("✅ Saved:", PIT_WHEN_PATH)

Conditional train rows: 16892 Conditional test rows: 4623
Class weight sample: [(0, 0.8358106423990999), (1, 0.92164072953737), (2, 0.9695596578151959), (3, 0.9831914279132024), (4, 0.9930815191010489), (5, 1.0067833035688796), (6, 1.0159247983625328), (7, 1.060207053788306), (8, 1.0919373881658423), (9, 1.1218634793485216)]
Epoch 1/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - acc: 0.1130 - loss: 2.2881 - val_acc: 0.1352 - val_loss: 2.2626 - learning_rate: 0.0010
Epoch 2/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - acc: 0.1404 - loss: 2.2408 - val_acc: 0.1423 - val_loss: 2.2564 - learning_rate: 0.0010
Epoch 3/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - acc: 0.1474 - loss: 2.2276 - val_acc: 0.1486 - val_loss: 2.2453 - learning_rate: 0.0010
Epoch 4/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - acc: 0.1547 - loss: 2.2065 - val_acc: 0.1506 - val_loss: 2.2461 - learning_rate: 0.0010
Epoch 5/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - acc: 0.1649 - loss: 2.1819 - val_acc: 0.1633 - val_loss:

In [ ]:
import numpy as np

K = K_WINDOW
PIT_CLASS = "y_pit_class"
PIT_BIN = "y_pit_within"

# cond_test exists from your training cell; otherwise rebuild it:
cond_test_eval = cond_test.copy()

def predict_cond(df_in):
    x = {}
    for c in CAT_COLS:
        x[c] = df_in[c].astype(str).values
    for n in NUM_COLS:
        x[n] = df_in[n].astype(np.float32).values
    return pit_when_model.predict(x, batch_size=4096, verbose=0)

probs = predict_cond(cond_test_eval)
y_true = cond_test_eval[PIT_CLASS].values.astype(int)
y_pred = probs.argmax(axis=1)

top1 = float((y_pred == y_true).mean())
top3 = float(np.mean([y_true[i] in np.argsort(probs[i])[-3:] for i in range(len(y_true))]))
mae_laps = float(np.mean(np.abs(y_pred - y_true)))

print("Conditional Top-1:", top1)
print("Conditional Top-3:", top3)
print("Conditional MAE (laps):", mae_laps)

Conditional Top-1: 0.16612589227774172
Conditional Top-3: 0.4308890330953926
Conditional MAE (laps): 2.7884490590525632
